In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error

In [2]:
train = pd.read_csv("challenge1_train.csv")
test = pd.read_csv("challenge1_test.csv")

print("Training data shape:", train.shape)
print("Test data shape:", test.shape)

Training data shape: (7398, 10)
Test data shape: (2467, 9)


In [4]:
display(train.head())

,ID,price,bathrooms,bedrooms,square_feet,latitude,longitude,category,has_photo,state
0,2,925,1.0,0,116,47.6160,-122.3275,housing/rent/apartment,Thumbnail,WA
1,3,2475,1.0,0,130,40.7629,-73.9885,housing/rent/apartment,Thumbnail,NY
2,5,1695,1.0,0,190,37.7599,-122.4379,housing/rent/apartment,Thumbnail,CA
3,6,1560,1.0,1,200,35.0847,-77.0609,housing/rent/apartment,Thumbnail,NC
4,7,1560,1.0,1,200,35.0960,-77.0272,housing/rent/apartment,Thumbnail,NC


In [5]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 7398 entries, 0 to 7397
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   ID           7398 non-null   int64  
 1   price        7398 non-null   int64  
 2   bathrooms    7398 non-null   float64
 3   bedrooms     7398 non-null   int64  
 4   square_feet  7398 non-null   int64  
 5   latitude     7398 non-null   float64
 6   longitude    7398 non-null   float64
 7   category     7398 non-null   str    
 8   has_photo    7398 non-null   str    
 9   state        7398 non-null   str    
dtypes: float64(3), int64(4), str(3)
memory usage: 578.1 KB


In [6]:
# Inspect categorical variables

print("Category values:")
print(train["category"].value_counts())

print("\nHas_photo values:")
print(train["has_photo"].value_counts())

print("\nNumber of states:")
print(train["state"].nunique())

print("\nState values:")
print(train["state"].value_counts())

Category values:
category
housing/rent/apartment     7395
housing/rent/home             2
housing/rent/short_term       1
Name: count, dtype: int64

Has_photo values:
has_photo
Thumbnail    6566
Yes           693
No            139
Name: count, dtype: int64

Number of states:
50

State values:
state
TX    1327
CA     688
WA     375
MD     341
NC     340
NJ     294
GA     271
FL     249
OH     232
CO     230
WI     222
IL     203
MO     189
MN     170
IN     165
VA     149
OK     143
MI     137
OR     135
IA     134
PA     130
MA     119
AZ      87
ND      87
NV      86
CT      78
NE      78
TN      73
UT      65
DC      62
SC      60
KS      60
NY      56
NH      48
LA      46
SD      46
AL      40
AR      40
AK      34
KY      30
ID      19
VT      14
HI      10
NM       9
RI       8
MS       7
MT       5
DE       4
ME       2
WV       1
Name: count, dtype: int64


In [7]:
# Summary statistics for numerical variables

display(train.describe().T)

,count,mean,std,min,25%,50%,75%,max
ID,7398.0,4954.080022,2845.314786,2.0000,2495.2500,4972.5000,7435.7500,9863.0000
price,7398.0,1462.805488,821.583589,200.0000,949.0000,1273.5000,1695.0000,8800.0000
bathrooms,7398.0,1.379832,0.602894,1.0000,1.0000,1.0000,2.0000,7.0000
bedrooms,7398.0,1.743850,0.936658,0.0000,1.0000,2.0000,2.0000,9.0000
square_feet,7398.0,940.935253,502.246805,116.0000,650.0000,807.5000,1100.0000,5921.0000
latitude,7398.0,37.647724,5.517342,21.3155,33.6047,38.7701,41.3498,61.5940
longitude,7398.0,-94.407727,15.722082,-158.0221,-98.5290,-93.6516,-81.6437,-70.3461


In [8]:
# Target variable summary

print("Price statistics:")
print(train["price"].describe())

print("\nNumber of unique prices:", train["price"].nunique())

Price statistics:
count    7398.000000
mean     1462.805488
std       821.583589
min       200.000000
25%       949.000000
50%      1273.500000
75%      1695.000000
max      8800.000000
Name: price, dtype: float64

Number of unique prices: 1498


## Feature and Target Preparation

The `ID` column is an identifier and is excluded from the predictors. The `price` column is the target variable. All remaining columns are used as potential predictors.

In [9]:
# Separate features and target

X = train.drop(columns=["price", "ID"])
y = train["price"]

X_test = test.drop(columns=["ID"])
test_ids = test["ID"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)

X shape: (7398, 8)
y shape: (7398,)
X_test shape: (2467, 8)


In [10]:
# Identify feature types

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

Numerical features: ['bathrooms', 'bedrooms', 'square_feet', 'latitude', 'longitude']
Categorical features: ['category', 'has_photo', 'state']


/var/folders/1m/tl03tj_53n3fk2hkbkvgp5wr0000gn/T/ipykernel_7232/2119906554.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))

Training samples: 5918
Validation samples: 1480


In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [13]:
from sklearn.metrics import root_mean_squared_error

def evaluate_model(model):
    model.fit(X_train, y_train)

    train_predictions = model.predict(X_train)
    val_predictions = model.predict(X_val)

    train_rmse = root_mean_squared_error(
        y_train,
        train_predictions
    )

    val_rmse = root_mean_squared_error(
        y_val,
        val_predictions
    )

    return train_rmse, val_rmse

## Baseline Linear Regression

We first train a Linear Regression model as a baseline. This provides a reference point for evaluating the more complex models.

In [16]:
from sklearn.linear_model import LinearRegression

linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

In [17]:
train_rmse, val_rmse = evaluate_model(linear_model)

print("Linear Regression")
print("Training RMSE:", train_rmse)
print("Validation RMSE:", val_rmse)

Linear Regression
Training RMSE: 515.3726439579439
Validation RMSE: 551.4034530433094
